# DataCo Late-Delivery Prediction
## Research background, target, and leakage prevention
This notebook predicts the order-level `Late_delivery_risk` target at order placement. It excludes canceled shipments and post-outcome fields. It runs unchanged with a local repository CSV or with a CSV uploaded in Google Colab.


In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, confusion_matrix

SOURCE = Path('DataCo_Retail_Processed.csv')
if not SOURCE.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('DataCo_Retail_Processed.csv was not found in the working directory. Run from the repository root or upload it in Colab.') from exc
    uploaded = files.upload()
    if 'DataCo_Retail_Processed.csv' not in uploaded:
        raise FileNotFoundError('Upload the repository CSV named DataCo_Retail_Processed.csv.')
raw = pd.read_csv(SOURCE)
raw = raw[raw['Delivery Status'] != 'Shipping canceled'].copy()
assert raw.groupby('Order Id')['Late_delivery_risk'].nunique().max() == 1
df = raw.groupby('Order Id', as_index=False).agg(order_date=('Order Date','first'), late_delivery=('Late_delivery_risk','first'), shipping_mode=('Shipping Mode','first'), market=('Market','first'), region=('Order Region','first'), country=('Order Country','first'), customer_segment=('Customer Segment','first'), category=('Category Name',lambda x:' | '.join(sorted(set(x)))), quantity=('Order Item Quantity','sum'), order_value=('Sales','sum'))
df['order_date'] = pd.to_datetime(df['order_date'])
df['order_month'] = df['order_date'].dt.month
df = df.sort_values(['order_date','Order Id'], kind='stable').reset_index(drop=True)
print({'orders': len(df), 'late_rate': float(df.late_delivery.mean()), 'source': str(SOURCE)})


## Chronological split and leakage-safe features
The first 70% of orders train each pipeline, the next 15% validate model and threshold choices, and the latest 15% are used only for final historical disclosure. No identifier, actual delivery information, `Delivery Status`, or `Late_delivery_risk` is included as a feature.


In [ ]:
features = ['shipping_mode','market','region','country','customer_segment','category','quantity','order_value','order_month']
cat = features[:6]; num = features[6:]
n = len(df); split_train, split_validation = int(.7*n), int(.85*n)
train, val, test = df.iloc[:split_train], df.iloc[split_train:split_validation], df.iloc[split_validation:]
print(pd.DataFrame({'rows':[len(train),len(val),len(test)], 'late_rate':[train.late_delivery.mean(),val.late_delivery.mean(),test.late_delivery.mean()]}, index=['train','validation','test']))

def make_preprocessor(cols):
    categorical = [x for x in cat if x in cols]
    numeric = [x for x in num if x in cols]
    return ColumnTransformer([
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical),
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
    ])

def evaluate(pipeline, frame, threshold=.5):
    probability = pipeline.predict_proba(frame[features])[:, 1]
    predicted = probability >= threshold
    tn, fp, fn, tp = confusion_matrix(frame.late_delivery, predicted).ravel()
    return {'accuracy': accuracy_score(frame.late_delivery,predicted), 'precision': precision_score(frame.late_delivery,predicted,zero_division=0), 'recall': recall_score(frame.late_delivery,predicted,zero_division=0), 'f1': f1_score(frame.late_delivery,predicted,zero_division=0), 'roc_auc': roc_auc_score(frame.late_delivery,probability), 'average_precision': average_precision_score(frame.late_delivery,probability), 'tn':tn, 'fp':fp, 'fn':fn, 'tp':tp, 'alert_rate':float(predicted.mean())}

models = {'majority_baseline': DummyClassifier(strategy='most_frequent'), 'logistic_regression': LogisticRegression(max_iter=1000, random_state=42), 'decision_tree': DecisionTreeClassifier(max_depth=12, min_samples_leaf=30, random_state=42), 'random_forest': RandomForestClassifier(n_estimators=150, min_samples_leaf=15, n_jobs=-1, random_state=42)}
fitted, comparison = {}, []
for name, estimator in models.items():
    pipeline = Pipeline([('preprocess', make_preprocessor(features)), ('model', estimator)])
    pipeline.fit(train[features], train.late_delivery)
    fitted[name] = pipeline
    comparison.append({'model':name, **evaluate(pipeline, val)})
comparison = pd.DataFrame(comparison).set_index('model')
display(comparison)


## Threshold selection, final test disclosure, and shipping-mode robustness
The Random Forest is selected from validation model performance. Its decision threshold is chosen from validation rows only, subject to a 70% precision floor and a 10–70% alert-rate range. The test period is then evaluated once for historical disclosure.


In [ ]:
forest = fitted['random_forest']
validation_probability = forest.predict_proba(val[features])[:, 1]
threshold_rows = []
for threshold in np.round(np.arange(.05, 1.00, .05), 2):
    predicted = validation_probability >= threshold
    threshold_rows.append({'threshold':threshold, 'precision':precision_score(val.late_delivery,predicted,zero_division=0), 'recall':recall_score(val.late_delivery,predicted,zero_division=0), 'f1':f1_score(val.late_delivery,predicted,zero_division=0), 'alert_rate':float(predicted.mean())})
thresholds = pd.DataFrame(threshold_rows)
eligible = thresholds.query('precision >= .70 and alert_rate >= .10 and alert_rate <= .70')
selected_threshold = float(eligible.sort_values(['f1','threshold'], ascending=[False,False]).iloc[0].threshold)
print('Selected validation threshold:', selected_threshold)
display(thresholds)
print('Final historical test metrics')
display(pd.DataFrame([evaluate(forest, test, selected_threshold)], index=['random_forest']))

def robustness_result(cols):
    pipe = Pipeline([('preprocess', make_preprocessor(cols)), ('model', RandomForestClassifier(n_estimators=150, min_samples_leaf=15, n_jobs=-1, random_state=42))])
    pipe.fit(train[cols], train.late_delivery)
    probability = pipe.predict_proba(val[cols])[:, 1]
    predicted = probability >= .5
    return {'roc_auc':roc_auc_score(val.late_delivery, probability), 'precision':precision_score(val.late_delivery,predicted,zero_division=0), 'recall':recall_score(val.late_delivery,predicted,zero_division=0), 'f1':f1_score(val.late_delivery,predicted,zero_division=0), 'alert_rate':float(predicted.mean())}

robustness = pd.DataFrame({'full_approved_features':robustness_result(features), 'shipping_mode_only':robustness_result(['shipping_mode']), 'without_shipping_mode':robustness_result([x for x in features if x != 'shipping_mode'])}).T
display(robustness)


## Interpretation and limitations
The robustness experiment quantifies association, not causality. The chronological test period has been accessed in earlier work and is therefore disclosed as historical evaluation, not a newly independent holdout. A later unseen period, monitoring, calibration validation, and an operational alert-capacity policy are needed before deployment.
